# EMV PDF Extraction — Clean Pipeline (Stages 0 → 4)

This notebook implements a simple, auditable extraction pipeline for EMV PDFs using **pdfplumber**.

Stages included:
1. **Stage 0 — Profile & paths**
2. **Stage 1 — Page primitives** (words + lines)
3. **Stage 1.5 — Noise removal** (repeated headers/footers) → `clean_text`
4. **Stage 2 — Headings extraction** (regex + font size)
5. **Stage 3 — Tables extraction** (separate from body text)
6. **Stage 4 — Definitions extraction** (from definition sections + simple patterns)

In [1]:
# Stage 0 — Imports
import os
import re
import json
import math
import csv
from pathlib import Path
from collections import Counter, defaultdict

import pdfplumber
import pandas as pd


## Stage 0 — Profile & output folders
Set up Profile and environement

In [2]:
PROFILE = {
    "doc_id": "EMV_Book_1",
    "doc_title": "EMV Integrated Circuit Card Specifications for Payment Systems — Book 1",
    "doc_version": "v4.4",              # update
    "doc_pages": None,
    "doc_date": "2022-12",              # update (YYYY-MM)
    "pdf_path": "../../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf",  # <-- update

    "out_root": "../../../data/processed/book1/",

    # Line building
    "line_top_tol": 5.0,          # cluster words into same line if tops are within this many points
    "line_x_gap": 2.0,            # add a space if the next word starts > this many points after prev word ends

    # Noise removal
    "header_ratio": 0.087, 
    "footer_ratio": 0.14, 
    "header_h_ratio": 0.132,         
    "footer_h_ratio": 0.15,         

    "noise_min_repetition": 0.30, # line must repeat on >= 60% of pages to be removed
    # Heading detection
    "heading_patterns": [
        r"^\s*(PART|Part)\s+[IVXLCDM]+\b.*$",         # PART I
        r"^\s*\d+(\.\d+)*\s+.+$",           # 1 / 1.1 / 1.1.1 Title
        r"^\s*(ANNEX|Annex)\s+[A-Z]\b.*$",    # Annex A ...
        r"^\s*[A-Z]\d+(\.\d+){0,2}\s+.+$",    # A1/ A1.1/ A1.1.1 Title
        #r"^\s*(FIGURE|Figure|TABLE|Table)\s+\d+.*$",  # Table 1 / Figure 1 ... 

        
    ],

    # Definitions
    "definition_section_keywords": ["definitions", "terms and definitions", "terminology"],
    "definition_line_patterns": [
        r"^\s*([A-Za-z][A-Za-z0-9 /\-]{2,})\s*[:\-—]\s+(.+)$"
    ],
}

out_root = Path(PROFILE["out_root"])
(out_root / "pages").mkdir(parents=True, exist_ok=True)
(out_root / "pages" / "by_page").mkdir(parents=True, exist_ok=True)
(out_root / "tables").mkdir(parents=True, exist_ok=True)
(out_root / "headings").mkdir(parents=True, exist_ok=True)
(out_root / "definitions").mkdir(parents=True, exist_ok=True)

print("Output root:", out_root.resolve())


Output root: /app/src/data/processed/book1


In [3]:
def save_profile():
    profile_path = os.path.join(PROFILE["out_root"], "profile.json")
    with open(profile_path, "w", encoding="utf-8") as f:
        json.dump(PROFILE, f, ensure_ascii=False, indent=2)
save_profile()

# Stage 1 — Page Text Extraction (Words → Lines)

## What we do
- Extract **words** from each PDF page  
- Use word **position** and **font information**  
- Rebuild them into readable **text lines**
## Why
PDF text is not stored like a normal document.
It is saved as many small “word boxes”, each with:
- Coordinates on the page
- Font properties
- No real reading order

If we read it directly:

❌ Text appears broken  
❌ Sentences are scattered  
❌ Layout is lost  

This stage restores a natural reading structure so the document becomes usable.
## Technology Used

### pdfplumber

**What it is**  
A Python library specialized in reading and extracting content from PDFs.

**What we use it for**
- Reading PDF pages
- Extracting words with their:
  - Text content
  - Position on page (x, y coordinates)
  - Font size
  - Font name

**Key Function Used**

```python
page.extract_words()
```

**What this function returns**
A list of word dictionaries like:

[
  {
    "text": "Transaction",
    "x0": 72.3,
    "x1": 132.5,
    "top": 145.2,
    "bottom": 158.6,
    "fontname": "Helvetica-Bold",
    "size": 11.0
  }
]
## Output

For each page, we produce structured data:

**Text Content**

- Clean reconstructed text lines

- Natural reading flow

**Layout Metadata**

- Word positions

- Line bounding boxes

- Font size information

In [4]:
def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def _safe_text(t):
    if t is None:
        return ""
    return str(t).strip()


def _get_meta_subset(w, keys):
    meta = {}
    for k in keys:
        if k in w and w[k] is not None:
            meta[k] = w[k]
    return meta


def words_to_lines(words, y_tol=4.0, join_with=" "):
    if not words:
        return []

    ws = []
    for idx, w in enumerate(words):
        try:
            ws.append({
                "_idx": idx,  # preserve original order
                "text": _safe_text(w.get("text", "")),
                "x0": float(w.get("x0", 0.0)),
                "x1": float(w.get("x1", 0.0)),
                "top": float(w.get("top", 0.0)),
                "bottom": float(w.get("bottom", 0.0)),
                "meta": _get_meta_subset(w, ["size", "fontname", "upright", "direction", "doctop"]),
            })
        except Exception:
            continue

    if not ws:
        return []

    # IMPORTANT: no sorting here. We keep extractor order.
    row_clusters = []
    current = [ws[0]]
    current_top = ws[0]["top"]

    for w in ws[1:]:
        if abs(w["top"] - current_top) <= y_tol:
            current.append(w)
            current_top = (current_top * (len(current) - 1) + w["top"]) / len(current)
        else:
            row_clusters.append(current)
            current = [w]
            current_top = w["top"]

    row_clusters.append(current)

    lines = []

    for row in row_clusters:
        if not row:
            continue

        segments = []
        seg = [row[0]]

        for w in row[1:]:
            seg.append(w)

        segments.append(seg)

        for seg in segments:
            seg_text_parts = [g["text"] for g in seg if g["text"]]
            text = join_with.join(seg_text_parts).strip()

            if not text:
                continue

            x0 = min(g["x0"] for g in seg)
            x1 = max(g["x1"] for g in seg)
            top = min(g["top"] for g in seg)
            bottom = max(g["bottom"] for g in seg)

            sizes = [g["meta"].get("size") for g in seg if "size" in g["meta"]]

            meta = {}

            if sizes:
                try:
                    meta["avg_size"] = float(sum(sizes) / len(sizes))
                except Exception:
                    pass

            meta["segment_words"] = int(len(seg))

            lines.append({
                "text": text,
                "x0": x0,
                "x1": x1,
                "top": top,
                "bottom": bottom,
                "meta": meta,
                "word_count": len(seg),
            })

    return lines


def extract_page_words(page):
    return page.extract_words(
        keep_blank_chars=False,
        use_text_flow=True,
        extra_attrs=["fontname", "size"]
    )


### Run Stage 1 extraction
Outputs:
- `pages/pages_raw.jsonl`
- `pages/pages_summary.csv`
- `pages/by_page/page_XXXX.json`


In [5]:
from datetime import datetime

raw_jsonl_path = out_root / "pages" / "pages_raw.jsonl"
summary_csv_path = out_root / "pages" / "pages_summary.csv"
by_page_dir = out_root / "pages" / "by_page"

records = []
summ_rows = []

with pdfplumber.open(PROFILE["pdf_path"]) as pdf:
    PROFILE["doc_pages"] = len(pdf.pages)
    print("Pages:", len(pdf.pages))
    for i, page in enumerate(pdf.pages, start=1):
        words = extract_page_words(page)
        lines = words_to_lines(words, y_tol=PROFILE["line_top_tol"])
        rec = {
            "doc_id": PROFILE["doc_id"],
            "doc_version": PROFILE["doc_version"],
            "doc_date": PROFILE["doc_date"],
            "page_num": i,
            "page_width": page.width,
            "page_height": page.height,
            "words": words,
            "lines": lines,
            "raw_text": "\n".join([l["text"] for l in lines]),
            "extracted_at": datetime.utcnow().isoformat() + "Z",
        }
        records.append(rec)
        (by_page_dir / f"page_{i:04d}.json").write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        summ_rows.append({
            "page_num": i,
            "n_words": len(words),
            "n_lines": len(lines),
            "raw_text_chars": len(rec["raw_text"]),
        })
save_profile()
with raw_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

pd.DataFrame(summ_rows).to_csv(summary_csv_path, index=False, encoding="utf-8")

print("Wrote:", raw_jsonl_path)
print("Wrote:", summary_csv_path)


Pages: 81
Wrote: ../../../data/processed/book1/pages/pages_raw.jsonl
Wrote: ../../../data/processed/book1/pages/pages_summary.csv


# Stage 1.5 — Noise Removal (Headers & Footers)

## What we do
- Detect repeated lines that appear on many pages
- Remove them automatically
- Focus on real content only

## Why
PDFs contain repeating noise like:

- Book titles
- Page numbers
- Copyright text
- Section banners

Outputs:
- `pages/noise_map.json`
- `pages/removed_lines_by_page.json`
- `pages/pages_clean.jsonl`


In [6]:
def line_in_header_or_footer(line, page_h, header_ratio, footer_ratio):
    if line.get("top") is None or line.get("bottom") is None:
        return False
    header_y = page_h * header_ratio
    footer_y = page_h * (1 - footer_ratio)
    return (line["bottom"] <= header_y) or (line["top"] >= footer_y)

def build_noise_map(page_records, min_rep):
    n_pages = len(page_records)
    counter = Counter()

    for rec in page_records:
        h, w = rec["page_height"], rec["page_width"]
        # Determine which ratio to use based on orientation
        current_h_ratio = PROFILE["header_ratio"] if h > w else PROFILE["header_h_ratio"]
        current_f_ratio = PROFILE["footer_ratio"] if h > w else PROFILE["footer_h_ratio"]

        for ln in rec["lines"]:
            if line_in_header_or_footer(ln, h, current_h_ratio, current_f_ratio):
                t = normalize_ws(ln["text"]).lower().strip()
                if t:
                    counter[t] += 1

    threshold = math.ceil(n_pages * float(min_rep))
    remove = {t for t, c in counter.items()}

    return {
        "n_pages": n_pages,
        "threshold_pages": threshold,
        "remove_texts": sorted(list(remove)),
        "counts": dict(counter),
    }

noise_map = build_noise_map(
    records,
    min_rep=PROFILE["noise_min_repetition"]
)

noise_map_path = out_root / "pages" / "noise_map.json"
noise_map_path.write_text(json.dumps(noise_map, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", noise_map_path)
print("Remove candidates:", len(noise_map["remove_texts"]))


Wrote: ../../../data/processed/book1/pages/noise_map.json
Remove candidates: 120


In [7]:
clean_jsonl_path = out_root / "pages" / "pages_clean.jsonl"
remove_set = set(noise_map["remove_texts"])

def clean_page_lines(rec, remove_set):
    clean_lines = []
    removed = []
    for ln in rec["lines"]:
        t_norm = normalize_ws(ln["text"]).lower()
        h, w = rec["page_height"], rec["page_width"]
        current_h_ratio = PROFILE["header_ratio"] if h > w else PROFILE["header_h_ratio"]
        current_f_ratio = PROFILE["footer_ratio"] if h > w else PROFILE["footer_h_ratio"]
        if (
            t_norm in remove_set
            and line_in_header_or_footer(ln, h, current_h_ratio, current_f_ratio)
        ):
            removed.append(ln["text"])
            continue
        clean_lines.append(ln)
    return clean_lines, removed

clean_records = []
removed_log = []

for rec in records:
    clean_lines, removed = clean_page_lines(rec, remove_set)
    rec2 = dict(rec)
    rec2["clean_lines"] = clean_lines
    rec2["clean_text"] = "\n".join([l["text"] for l in clean_lines])
    clean_records.append(rec2)
    removed_log.append({"page_num": rec["page_num"], "removed_lines": removed})

with clean_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in clean_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

(out_root / "pages" / "removed_lines_by_page.json").write_text(
    json.dumps(removed_log, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Wrote:", clean_jsonl_path)
print("Wrote:", out_root / "pages" / "removed_lines_by_page.json")


Wrote: ../../../data/processed/book1/pages/pages_clean.jsonl
Wrote: ../../../data/processed/book1/pages/removed_lines_by_page.json


# Stage 2 — Headings Extraction (Document Structure)

## What we do
Identify section titles using:

- Regex patterns  
- Font size (bigger text = likely heading)

Then:
- Order headings properly
- Build hierarchy (parent → child sections)
- Detect section spans across pages

## Why
Headings define the document structure:


Book
└── Section
└── Subsection
└── Paragraphs

This structure is critical for:
- Navigation  
- Section-level search  
- Accurate citations   
## Output
Structured heading records:
`headings/headings.json`

Each heading includes:
- Title text
- Level (section/subsection)
- Page location
- Parent section
- Covered page range


# Heading Processing — Function Summaries
### `looks_like_heading(...)`
Detects if a text line is likely a heading.  
It checks heading patterns and ensures the font size is large enough.  
Returns True/False with the detection reason.
### `infer_heading_level(...)`
Determines the hierarchy level of a heading from its numbering style.  
Recognizes Parts, Annexes, Sections, and nested subsections.  
Returns a numeric level (0 = highest level).
### `should_merge_heading_lines(...)`
Checks if two consecutive lines are parts of the same heading.  
Uses font size similarity, vertical distance, and heading patterns.  
Returns True if they should be merged.
### `merge_wrapped_heading_lines(...)`
Merges multi-line headings into a single complete heading.  
Combines text and updates position and font metadata.  
Returns a cleaned list of properly formed headings.

In [8]:
heading_res = [re.compile(p) for p in PROFILE["heading_patterns"]]

def looks_like_heading(text, font_avg, patterns):
    t = text.strip()
    if not t:
        return False, None

    matches_pattern = any(p.match(t) for p in patterns)
    has_large_font = False

    if font_avg is not None :
        if font_avg >= 12:
            has_large_font = True

    if matches_pattern and has_large_font and len(t) <= 160:
        return True, "regex_and_font"

    return False, None


def infer_heading_level(text):
    RE_LEVEL_4 = re.compile(r"^\s*([A-Z])\d+\.\d+\.\d+\s+.+$")
    RE_LEVEL_3 = re.compile(r"^\s*([A-Z])\d+\.\d+\s+.+$")
    RE_LEVEL_2 = re.compile(r"^\s*([A-Z])\d+\s+.+$")
    RE_NUMERIC = re.compile(r"^\s*(\d+(?:\.\d+)*)\s+.+$")

    t = text.strip()

    if re.match(r"^\s*(PART|Part)\s+[IVXLCDM]+\b.*$", t):
        return 0
    if re.match(r"^\s*(ANNEX|Annex)\s+[A-Z]\b.*$", t):
        return 1

    if RE_LEVEL_4.match(t):
        return 4
    if RE_LEVEL_3.match(t):
        return 3
    if RE_LEVEL_2.match(t):
        return 2

    m = RE_NUMERIC.match(t)
    if m:
        depth = m.group(1).count(".") + 1
        return min(depth, 6)

    return 2

def should_merge_heading_lines(curr, nxt, max_gap=50.0, size_tol=0.8):
    if not curr or not nxt:
        return False

    curr_text = curr["text"].strip()
    nxt_text = nxt["text"].strip()

    if not curr_text or not nxt_text:
        return False

    curr_size = curr.get("meta", {}).get("avg_size")
    nxt_size = nxt.get("meta", {}).get("avg_size")

    if curr_size is None or nxt_size is None:
        return False

    # similar font sizes
    if curr_size < 20 and nxt_size < 20:
        if abs(curr_size - nxt_size) > size_tol:
            return False
    elif curr_size >= 20 and nxt_size >= 20:
        if abs(curr_size - nxt_size) > 20:
            return False
    else:
        return False

    # heading-like font size
    if curr_size < 12 or nxt_size < 12:
        return False

    # vertically close
    curr_bottom = curr.get("bottom")
    nxt_top = nxt.get("top")
    if curr_bottom is None or nxt_top is None:
        return False

    if (nxt_top - curr_bottom) > max_gap:
        return False

    # current merged text must still look like a heading start
    first_is_heading, _ = looks_like_heading(
        curr_text,
        curr_size,
        heading_res
    )
    if not first_is_heading:
        return False

    # next line should not itself be a new standalone heading
    if any(p.match(nxt_text) for p in heading_res):
        return False

    return True


def merge_wrapped_heading_lines(lines, max_line_merge=3):
    merged = []
    i = 0

    while i < len(lines):
        curr = dict(lines[i])
        merged_count = 1
        j = i + 1

        while j < len(lines) and merged_count < max_line_merge:
            nxt = lines[j]

            if not should_merge_heading_lines(curr, nxt):
                break

            curr = {
                "text": f"{curr['text'].rstrip()} {nxt['text'].lstrip()}",
                "x0": min(curr.get("x0", 0), nxt.get("x0", 0)),
                "x1": max(curr.get("x1", 0), nxt.get("x1", 0)),
                "top": min(curr.get("top", 0), nxt.get("top", 0)),
                "bottom": max(curr.get("bottom", 0), nxt.get("bottom", 0)),
                "meta": {
                    "avg_size": max(
                        curr.get("meta", {}).get("avg_size", 0),
                        nxt.get("meta", {}).get("avg_size", 0)
                    )
                }
            }

            merged_count += 1
            j += 1

        merged.append(curr)
        i = j

    return merged

### `build_heading_ids_and_parents(...)`
Sorts headings in document order and gives each one a unique heading ID.  
Uses heading levels to find the parent heading of each section.  
Returns an enriched heading list with `heading_id` and `parent_heading_id`.

In [9]:
def build_heading_ids_and_parents(headings):
    # Ensure document order
    headings_sorted = sorted(
        headings,
        key=lambda h: (h["page_num"], h["bbox"]["top"] if h.get("bbox") else 0)
    )
    enriched = []
    stack = []  # keeps last seen headings by hierarchy

    for idx, h in enumerate(headings_sorted, start=1):
        h = dict(h)
        h["heading_id"] = (
            f"{h['doc_id']}_{h['doc_version']}"
            f"_p{h['page_num']:04d}_h{idx:04d}"
        )
        while stack and stack[-1]["level"] >= h["level"]:
            stack.pop()

        if stack:
            h["parent_heading_id"] = stack[-1]["heading_id"]
        else:
            h["parent_heading_id"] = None

        stack.append(h)
        enriched.append(h)

    return enriched

### `add_heading_spans(...)`
Determines the content range covered by each heading in the document.  
It finds where a heading starts and where it ends based on the next heading or page limits.  
Returns headings enriched with span info (`start_page/top`, `end_page/top`).

In [10]:
def add_heading_spans(headings, clean_records, last_page_num, min_text_len=3, footer_ratio=0.145):
    if not headings:
        return headings

    hs = sorted(
        headings,
        key=lambda h: (h["page_num"], h["bbox"]["top"] if h.get("bbox") else 0)
    )

    page_lines_map = {
        rec["page_num"]: rec.get("clean_lines", [])
        for rec in clean_records
    }

    page_height_map = {
        rec["page_num"]: rec.get("page_height")
        for rec in clean_records
    }

    def last_text_above_heading(page_num, heading_top):
        lines = page_lines_map.get(page_num, [])
        candidates = []

        for ln in lines:
            text = (ln.get("text") or "").strip()
            line_top = ln.get("top")
            line_bottom = ln.get("bottom")

            if not text or len(text) < min_text_len:
                continue
            if line_top is None or line_bottom is None:
                continue

            if line_bottom <= heading_top:
                candidates.append(ln)

        if not candidates:
            return None

        return max(candidates, key=lambda ln: ln.get("bottom", 0))

    def usable_page_bottom(page_num):
        page_h = page_height_map.get(page_num)
        if page_h is None:
            return None
        return page_h * (1 - footer_ratio)

    def find_next_boundary(idx):
        """
        Find the next heading that closes the current heading span.
        Rule: next heading with level <= current level.
        """
        curr_level = hs[idx]["level"]

        for j in range(idx + 1, len(hs)):
            if hs[j]["level"] <= curr_level:
                return hs[j]

        return None

    for i, h in enumerate(hs):
        h["start_page"] = h["page_num"]
        h["start_top"] = h["bbox"]["top"] if h.get("bbox") else None

        boundary_h = find_next_boundary(i)

        if boundary_h is not None:
            boundary_page = boundary_h["page_num"]
            boundary_top = boundary_h["bbox"]["top"] if boundary_h.get("bbox") else None

            if boundary_top is not None:
                last_line = last_text_above_heading(boundary_page, boundary_top)
            else:
                last_line = None

            if last_line is not None:
                h["end_page"] = boundary_page
                h["end_top"] = last_line.get("bottom")
            else:
                prev_page = max(h["start_page"], boundary_page - 1)
                h["end_page"] = prev_page
                h["end_top"] = usable_page_bottom(prev_page)
        else:
            h["end_page"] = last_page_num
            h["end_top"] = usable_page_bottom(last_page_num)

    return hs

This section scans all cleaned pages, detects lines that look like headings, and extracts their section number, title, level, and position.  
Then it adds parent-child links and the span covered by each heading, and saves everything into `headings.json`.  

### 📤 Output Example (`headings.json`)
```json
[
  {
    "doc_id": "EMV_BOOK3",
    "doc_version": "4.4",
    "doc_date": "2024-01-01",
    "page_num": 12,
    "section_number": "1.2",
    "title": "Application Selection",
    "level": 2,
    "reason": "regex_and_font",
    "bbox": {
      "x0": 72.1,
      "top": 145.3,
      "x1": 310.8,
      "bottom": 160.4
    },
    "font_size_avg": 14.0,
    "heading_id": "EMV_BOOK3_4.4_p0012_h0003",
    "parent_heading_id": "EMV_BOOK3_4.4_p0010_h0001",
    "start_page": 12,
    "start_top": 145.3,
    "end_page": 14,
    "end_top": 620.5
  }
]

In [11]:
headings = []
pattern_split = re.compile(r"^\s*(PART\s+[IVX]+|ANNEX\s+[A-Z]|[A-Z]?\d+(?:\.\d+)*)\s+(.+)$", re.I)
for rec in clean_records:
    lines = rec["clean_lines"]
    
    lines_for_heading = merge_wrapped_heading_lines(lines)

    for ln in lines_for_heading:
        ok, why = looks_like_heading(
            ln["text"],
            ln["meta"].get("avg_size"),
            heading_res
        )
        if ok:
            match = pattern_split.match(ln["text"])
            raw_title = match.group(2).strip()
            cleaned = re.sub(r"[^a-zA-Z0-9\s\-\(\)\,\.]", "", match.group(2))
            cleaned_title = re.sub(r"^[^a-zA-Z0-9]+", "", cleaned).strip()
            headings.append({
                "doc_id": rec["doc_id"],
                "doc_version": rec["doc_version"],
                "doc_date": rec["doc_date"],
                "page_num": rec["page_num"],
                "section_number": match.group(1),
                "title": cleaned_title,
                "level": infer_heading_level(ln["text"]),
                "reason": why,
                "bbox": {
                    "x0": ln.get("x0"),
                    "top": ln.get("top"),
                    "x1": ln.get("x1"),
                    "bottom": ln.get("bottom")
                },
                "font_size_avg": ln["meta"].get("avg_size"),            })
headings = build_heading_ids_and_parents(headings)
last_page = clean_records[-1]["page_num"]

headings = add_heading_spans(
    headings=headings,
    clean_records=clean_records,
    last_page_num=PROFILE["doc_pages"]
)
headings_path = out_root / "headings" / "headings.json"
headings_path.write_text(json.dumps(headings, ensure_ascii=False, indent=2), encoding="utf-8")

print("Headings:", len(headings))
print("Wrote:", headings_path)

Headings: 68
Wrote: ../../../data/processed/book1/headings/headings.json
